# Support Vector Machine(SVM)
This Notebook walks through building, tuning, and evaluating a Support Vector Machine classifier using 'scikit-learn'. Because SVMs are sensitive to feature scales, we incorporate a 'StandardScaler' inside a scikit-learn 'Pipeline' to prevent data leakage during cross-validation. 

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

RANDOM_STATE = 42

## 1. Load and Prepare the Data
Replace the synthetic dataset below with your actual pandas DataFrame or numpy arrays (`X` and `y`).

In [ ]:
df = pd.read_csv("../Data/supermarket_subset.csv")

display(df.head())

target_column = 'repeat_purchase'

X = df.drop(columns=[target_column])

y = df[target_column]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=RANDOM_STATE, 
    stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

FileNotFoundError: [Errno 2] No such file or directory: 'supermarket_subset.csv'

## 2. Build the Pipeline & Tune Hyperparameters
We combine `StandardScaler` and `SVC` into a single pipeline. We then use `GridSearchCV` to search for the optimal combination of:
- **`C`**: Regularization parameter.
- **`kernel`**: Decision boundary type (`linear` vs. `rbf`).
- **`gamma`**: Kernel coefficient for the RBF kernel.

In [ ]:
# Create pipeline chaining scaling and SVM
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(random_state=RANDOM_STATE))
])

# Define hyperparameter grid to search
param_grid = {
    'svm__C': [0.1, 1, 10, 100],
    'svm__kernel': ['linear', 'rbf'],
    'svm__gamma': ['scale', 'auto', 0.01, 0.1, 1]
}

# Setup GridSearchCV with 5-fold cross-validation
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

# Fit the grid search on the training data
print("Running Grid Search for optimal SVM hyperparameters...")
grid_search.fit(X_train, y_train)

# Display best parameters
print("\nBest Parameters Found:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")

Running Grid Search for optimal SVM hyperparameters...


NameError: name 'X_train' is not defined

## 3. Evaluate the Best Model
Let's evaluate our tuned model on the unseen test set using classification metrics and a confusion matrix.

In [ ]:
# Extract best estimator from grid search
best_model = grid_search.best_estimator_

# Predict on test data
y_pred = best_model.predict(X_test)

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Plot Confusion Matrix
plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap=plt.cm.Blues, values_format='d', ax=plt.gca())
plt.title("SVM Confusion Matrix")
plt.grid(False)
plt.show()

## 4. Optional: Visualizing Decision Boundaries (2D Feature Space)
If you are working with 2D data (or project your features down using PCA), you can visualize how the SVM draws its decision boundaries.

In [ ]:
# Train a 2D model specifically for visualization purposes
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_train_2d = pca.fit_transform(X_train)

svm_2d = SVC(kernel='rbf', C=grid_search.best_params_['svm__C'], gamma=grid_search.best_params_['svm__gamma'])
scaler_2d = StandardScaler()
X_train_2d_scaled = scaler_2d.fit_transform(X_train_2d)
svm_2d.fit(X_train_2d_scaled, y_train)

# Plot decision boundary
x_min, x_max = X_train_2d_scaled[:, 0].min() - 1, X_train_2d_scaled[:, 0].max() + 1
y_min, y_max = X_train_2d_scaled[:, 1].min() - 1, X_train_2d_scaled[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                     np.arange(y_min, y_max, 0.02))

Z = svm_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, cmap=plt.cm.coolwarm, alpha=0.8)
plt.scatter(X_train_2d_scaled[:, 0], X_train_2d_scaled[:, 1], c=y_train, cmap=plt.cm.coolwarm, edgecolors='k')
plt.title("SVM Decision Boundary (PCA-reduced 2D view)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.show()